In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas kagglehub

import os
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw5_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 133.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 117.7 MB/s eta 0:00:00
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [7]:
!pip install -q jupysql

%reload_ext sql
%sql engine

%config SqlMagic.displaylimit = 20


In [4]:
from urllib.request import urlretrieve

URL = (
    'https://d37ci6vzurychx.cloudfront.net/trip-data/'
    'yellow_tripdata_2024-03.parquet'
)

parquet_path = '/content/yellow_tripdata_2024-03.parquet'
urlretrieve(URL, parquet_path)

columns = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'trip_distance',
    'fare_amount',
    'passenger_count',
]

df = pd.read_parquet(parquet_path, columns=columns).head(500_000)

# Basic cleaning before loading to PostgreSQL
df = df[(df['fare_amount'] > 0) & (df['fare_amount'] < 500)]
df = df[(df['trip_distance'] > 0) & (df['trip_distance'] < 100)]
df = df[df['PULocationID'].notna()]

df.to_sql(
    'nyc_taxi_raw',
    engine,
    if_exists='replace',
    index=False,
    chunksize=10_000,
)

pd.read_sql('SELECT COUNT(*) AS n_rows FROM nyc_taxi_raw', engine)

,n_rows
0,486964


In [8]:
%%sql

DROP TABLE IF EXISTS t7_hw_taxi_daily;

CREATE TABLE t7_hw_taxi_daily AS
SELECT
    "PULocationID"::INTEGER AS pickup_zone_id,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE AS ts_day,
    COUNT(*)::INTEGER AS trip_count,
    AVG(fare_amount)::NUMERIC(10, 2) AS avg_fare,
    SUM(trip_distance)::NUMERIC(12, 2) AS total_distance
FROM nyc_taxi_raw
WHERE fare_amount > 0
  AND fare_amount < 500
  AND trip_distance > 0
  AND trip_distance < 100
  AND "PULocationID" IS NOT NULL
GROUP BY
    "PULocationID"::INTEGER,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE;

ALTER TABLE t7_hw_taxi_daily
    ADD PRIMARY KEY (pickup_zone_id, ts_day);

SELECT pickup_zone_id, COUNT(*) AS n_days
FROM t7_hw_taxi_daily
GROUP BY pickup_zone_id
ORDER BY pickup_zone_id
LIMIT 10;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

1217 rows affected.

10 rows affected.

pickup_zone_id,n_days
1,5
3,6
4,6
6,4
7,6
8,1
9,5
10,6
11,4
12,5


In [17]:
%%sql

SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    LAG(trip_count, 1)  OVER w AS trips_lag_1d,
    LAG(trip_count, 7)  OVER w AS trips_lag_7d,
    LAG(trip_count, 30) OVER w AS trips_lag_30d
FROM t7_hw_taxi_daily
WINDOW w AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

20 rows affected.

pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
1,2024-03-01,1,None,None,None
1,2024-03-02,2,1,None,None
1,2024-03-03,2,2,None,None
1,2024-03-04,1,2,None,None
1,2024-03-05,1,1,None,None
3,2024-03-01,3,None,None,None
3,2024-03-02,1,3,None,None
3,2024-03-03,4,1,None,None
3,2024-03-04,2,4,None,None
3,2024-03-05,1,2,None,None


In [18]:
%%sql

SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
    COUNT(*)      OVER w7 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w7 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

20 rows affected.

pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_in_window
1,2024-03-01,105.00,105.0000000000000000,1
1,2024-03-02,95.00,100.0000000000000000,2
1,2024-03-03,51.00,83.6666666666666667,3
1,2024-03-04,40.00,72.7500000000000000,4
1,2024-03-05,120.00,82.2000000000000000,5
3,2024-03-01,39.17,39.1700000000000000,1
3,2024-03-02,21.50,30.3350000000000000,2
3,2024-03-03,25.03,28.5666666666666667,3
3,2024-03-04,39.15,31.2125000000000000,4
3,2024-03-05,45.50,34.0700000000000000,5


In [19]:
%%sql

SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
    COUNT(*)              OVER w30 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w30 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

20 rows affected.

pickup_zone_id,ts_day,avg_fare,rolling_std_fare_30d,days_in_window
1,2024-03-01,105.00,None,1
1,2024-03-02,95.00,7.0710678118654752,2
1,2024-03-03,51.00,28.7286152352203244,3
1,2024-03-04,40.00,32.0455405113826511,4
1,2024-03-05,120.00,34.8812270426371325,5
3,2024-03-01,39.17,None,1
3,2024-03-02,21.50,12.4945768235662948,2
3,2024-03-03,25.03,9.3508413168726873,3
3,2024-03-04,39.15,9.2894505578460703,4
3,2024-03-05,45.50,10.2736045281098883,5


In [20]:
%%sql

SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    SUM(trip_count) OVER we AS cumulative_trips,
    AVG(trip_count) OVER we AS expanding_avg_trips
FROM t7_hw_taxi_daily
WINDOW we AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

20 rows affected.

pickup_zone_id,ts_day,trip_count,cumulative_trips,expanding_avg_trips
1,2024-03-01,1,1,1.00000000000000000000
1,2024-03-02,2,3,1.5000000000000000
1,2024-03-03,2,5,1.6666666666666667
1,2024-03-04,1,6,1.5000000000000000
1,2024-03-05,1,7,1.4000000000000000
3,2024-03-01,3,3,3.0000000000000000
3,2024-03-02,1,4,2.0000000000000000
3,2024-03-03,4,8,2.6666666666666667
3,2024-03-04,2,10,2.5000000000000000
3,2024-03-05,1,11,2.2000000000000000


In [22]:
%%sql

CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS
WITH features_basic AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        avg_fare,
        total_distance,

        LAG(trip_count, 1)  OVER w_pid AS trips_lag_1d,
        LAG(trip_count, 7)  OVER w_pid AS trips_lag_7d,
        LAG(trip_count, 30) OVER w_pid AS trips_lag_30d,

        AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
        STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,

        SUM(trip_count) OVER we AS cumulative_trips,
        AVG(trip_count) OVER we AS expanding_avg_trips
    FROM t7_hw_taxi_daily
    WINDOW
        w_pid AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
        ),
        w7 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
        ),
        w30 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
        ),
        we AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
)
SELECT *
FROM features_basic
WHERE trips_lag_1d IS NOT NULL;

SELECT *
FROM v_t7_hw_taxi_features
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

20 rows affected.

pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips
1,2024-03-02,2,95.00,18.43,1,None,None,100.0000000000000000,7.0710678118654752,3,1.5000000000000000
1,2024-03-03,2,51.00,0.09,2,None,None,83.6666666666666667,28.7286152352203244,5,1.6666666666666667
1,2024-03-04,1,40.00,0.01,2,None,None,72.7500000000000000,32.0455405113826511,6,1.5000000000000000
1,2024-03-05,1,120.00,0.03,1,None,None,82.2000000000000000,34.8812270426371325,7,1.4000000000000000
3,2024-03-02,1,21.50,1.80,3,None,None,30.3350000000000000,12.4945768235662948,4,2.0000000000000000
3,2024-03-03,4,25.03,24.13,1,None,None,28.5666666666666667,9.3508413168726873,8,2.6666666666666667
3,2024-03-04,2,39.15,16.86,4,None,None,31.2125000000000000,9.2894505578460703,10,2.5000000000000000
3,2024-03-05,1,45.50,13.90,2,None,None,34.0700000000000000,10.2736045281098883,11,2.2000000000000000
3,2024-03-06,1,53.50,0.80,1,None,None,37.3083333333333333,12.1391258608956958,12,2.0000000000000000
4,2024-03-02,273,15.84,729.24,108,None,None,16.1100000000000000,0.38183766184073566318,381,190.5000000000000000


In [23]:
%%sql

DROP TABLE IF EXISTS t7_hw_events;

CREATE TABLE t7_hw_events (
    event_id    INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    user_id     INTEGER NOT NULL,
    event_ts    TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    event_type  TEXT NOT NULL,
    payload     JSONB NOT NULL
);

INSERT INTO t7_hw_events (user_id, event_ts, event_type, payload)
SELECT
    (n % 1000) + 1 AS user_id,
    TIMESTAMPTZ '2024-04-01 00:00:00+00'
        - ((n % 60) || ' days')::INTERVAL AS event_ts,
    CASE (n % 5)
        WHEN 0 THEN 'page_view'
        WHEN 1 THEN 'click'
        WHEN 2 THEN 'add_to_cart'
        WHEN 3 THEN 'checkout'
        ELSE        'recommendation_shown'
    END AS event_type,
    CASE (n % 5)
        WHEN 0 THEN jsonb_build_object(
            'url', '/product/' || ((n % 200) + 1),
            'referrer', (ARRAY['google','direct','email','fb'])[(n % 4) + 1],
            'device', (ARRAY['mobile','desktop','tablet'])[(n % 3) + 1]
        )
        WHEN 1 THEN jsonb_build_object(
            'element_id', 'btn_' || ((n % 50) + 1),
            'page_url', '/category/' || ((n % 20) + 1),
            'device', (ARRAY['mobile','desktop'])[(n % 2) + 1]
        )
        WHEN 2 THEN jsonb_build_object(
            'product_id', (n % 200) + 1,
            'price', ((n % 100) + 10)::NUMERIC(10, 2),
            'quantity', (n % 5) + 1,
            'tags', jsonb_build_array(
                (ARRAY['promo','new','clearance'])[(n % 3) + 1]
            )
        )
        WHEN 3 THEN jsonb_build_object(
            'order_id', 'o_' || (n + 100000),
            'total', ((n % 500) + 50)::NUMERIC(10, 2),
            'items_count', (n % 10) + 1,
            'payment_method', (ARRAY['card','paypal','crypto'])[(n % 3) + 1]
        )
        ELSE jsonb_build_object(
            'model_name', 'rec_v' || ((n % 5) + 1),
            'model_version', '1.' || (n % 10),
            'recommendations', jsonb_build_array(
                (n % 200) + 1,
                ((n + 1) % 200) + 1,
                ((n + 2) % 200) + 1
            ),
            'experiment', 'exp_' || ((n % 3) + 1)
        )
    END AS payload
FROM generate_series(1, 50000) AS n;

ANALYZE t7_hw_events;

SELECT event_type, COUNT(*) AS n_events
FROM t7_hw_events
GROUP BY event_type
ORDER BY event_type;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

50000 rows affected.

5 rows affected.

event_type,n_events
add_to_cart,10000
checkout,10000
click,10000
page_view,10000
recommendation_shown,10000


In [24]:
%%sql

SELECT event_id, user_id, event_ts, payload
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND payload @> jsonb_build_object('payment_method', 'card')
LIMIT 10;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

10 rows affected.

event_id,user_id,event_ts,payload
3,4,2024-03-29 00:00:00+00:00,"{'total': 53.0, 'order_id': 'o_100003', 'items_count': 4, 'payment_method': 'card'}"
18,19,2024-03-14 00:00:00+00:00,"{'total': 68.0, 'order_id': 'o_100018', 'items_count': 9, 'payment_method': 'card'}"
33,34,2024-02-28 00:00:00+00:00,"{'total': 83.0, 'order_id': 'o_100033', 'items_count': 4, 'payment_method': 'card'}"
48,49,2024-02-13 00:00:00+00:00,"{'total': 98.0, 'order_id': 'o_100048', 'items_count': 9, 'payment_method': 'card'}"
63,64,2024-03-29 00:00:00+00:00,"{'total': 113.0, 'order_id': 'o_100063', 'items_count': 4, 'payment_method': 'card'}"
78,79,2024-03-14 00:00:00+00:00,"{'total': 128.0, 'order_id': 'o_100078', 'items_count': 9, 'payment_method': 'card'}"
93,94,2024-02-28 00:00:00+00:00,"{'total': 143.0, 'order_id': 'o_100093', 'items_count': 4, 'payment_method': 'card'}"
108,109,2024-02-13 00:00:00+00:00,"{'total': 158.0, 'order_id': 'o_100108', 'items_count': 9, 'payment_method': 'card'}"
123,124,2024-03-29 00:00:00+00:00,"{'total': 173.0, 'order_id': 'o_100123', 'items_count': 4, 'payment_method': 'card'}"
138,139,2024-03-14 00:00:00+00:00,"{'total': 188.0, 'order_id': 'o_100138', 'items_count': 9, 'payment_method': 'card'}"


In [25]:
%%sql

SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ? 'tags'
LIMIT 10;

SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ?| ARRAY['experiment', 'model_version']
LIMIT 10;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

10 rows affected.

10 rows affected.

event_id,event_type,payload
4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [5, 6, 7]}"
9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [10, 11, 12]}"
14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [15, 16, 17]}"
19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [20, 21, 22]}"
24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [25, 26, 27]}"
29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [30, 31, 32]}"
34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [35, 36, 37]}"
39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [40, 41, 42]}"
44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [45, 46, 47]}"
49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [50, 51, 52]}"


In [26]:
%%sql

SELECT
    event_id,
    jsonb_path_query(payload, '$.recommendations[*]') AS recommended_pid
FROM t7_hw_events
WHERE event_type = 'recommendation_shown'
LIMIT 15;

SELECT
    event_id,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND jsonb_path_exists(payload, '$.total ? (@ > 300)')
LIMIT 10;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

15 rows affected.

10 rows affected.

event_id,order_id,total
253,o_100253,303.00
258,o_100258,308.00
263,o_100263,313.00
268,o_100268,318.00
273,o_100273,323.00
278,o_100278,328.00
283,o_100283,333.00
288,o_100288,338.00
293,o_100293,343.00
298,o_100298,348.00


In [27]:
%%sql

DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;
DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
DROP INDEX IF EXISTS idx_t7_events_device_btree;

ANALYZE t7_hw_events;

EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card');

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

10 rows affected.

QUERY PLAN
Aggregate (cost=1756.60..1756.61 rows=1 width=8) (actual time=29.223..29.225 rows=1 loops=1)
Buffers: shared hit=1000
-> Seq Scan on t7_hw_events (cost=0.00..1750.00 rows=2639 width=0) (actual time=0.009..29.058 rows=3334 loops=1)
"Filter: (payload @> jsonb_build_object('payment_method', 'card'))"
Rows Removed by Filter: 46666
Buffers: shared hit=1000
Planning:
Buffers: shared hit=5
Planning Time: 0.126 ms
Execution Time: 29.244 ms


In [28]:
%%sql

CREATE INDEX idx_t7_events_payload_gin_pathops
    ON t7_hw_events USING GIN (payload jsonb_path_ops);

ANALYZE t7_hw_events;

SELECT pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_pathops'))
       AS gin_index_size;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

1 rows affected.

gin_index_size
1144 kB


In [29]:
%%sql

EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card');


Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

13 rows affected.

QUERY PLAN
Aggregate (cost=1098.98..1098.99 rows=1 width=8) (actual time=10.171..10.172 rows=1 loops=1)
Buffers: shared hit=1004
-> Bitmap Heap Scan on t7_hw_events (cost=35.88..1089.97 rows=3606 width=0) (actual time=0.612..9.856 rows=3334 loops=1)
"Recheck Cond: (payload @> jsonb_build_object('payment_method', 'card'))"
Heap Blocks: exact=1000
Buffers: shared hit=1004
-> Bitmap Index Scan on idx_t7_events_payload_gin_pathops (cost=0.00..34.98 rows=3606 width=0) (actual time=0.462..0.463 rows=3334 loops=1)
"Index Cond: (payload @> jsonb_build_object('payment_method', 'card'))"
Buffers: shared hit=4
Planning:


In [30]:
%%sql

EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('referrer', 'google');

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

13 rows affected.

QUERY PLAN
Aggregate (cost=1083.97..1083.98 rows=1 width=8) (actual time=3.698..3.699 rows=1 loops=1)
Buffers: shared hit=1004
-> Bitmap Heap Scan on t7_hw_events (cost=32.42..1076.61 rows=2946 width=0) (actual time=0.442..3.567 rows=2500 loops=1)
"Recheck Cond: (payload @> jsonb_build_object('referrer', 'google'))"
Heap Blocks: exact=1000
Buffers: shared hit=1004
-> Bitmap Index Scan on idx_t7_events_payload_gin_pathops (cost=0.00..31.68 rows=2946 width=0) (actual time=0.300..0.301 rows=2500 loops=1)
"Index Cond: (payload @> jsonb_build_object('referrer', 'google'))"
Buffers: shared hit=4
Planning:


In [31]:
%%sql

EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile';

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

8 rows affected.

QUERY PLAN
Aggregate (cost=1750.62..1750.63 rows=1 width=8) (actual time=8.110..8.111 rows=1 loops=1)
Buffers: shared hit=1000
-> Seq Scan on t7_hw_events (cost=0.00..1750.00 rows=250 width=0) (actual time=0.021..7.697 rows=8333 loops=1)
Filter: ((payload ->> 'device'::text) = 'mobile'::text)
Rows Removed by Filter: 41667
Buffers: shared hit=1000
Planning Time: 0.086 ms
Execution Time: 8.163 ms


In [32]:
%%sql

CREATE INDEX idx_t7_events_device_btree
    ON t7_hw_events ((payload ->> 'device'));

ANALYZE t7_hw_events;

EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile';

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

13 rows affected.

QUERY PLAN
Aggregate (cost=1242.29..1242.30 rows=1 width=8) (actual time=1.276..1.277 rows=1 loops=1)
Buffers: shared hit=2 read=9
-> Bitmap Heap Scan on t7_hw_events (cost=96.75..1221.50 rows=8317 width=0) (actual time=0.401..0.906 rows=8333 loops=1)
Recheck Cond: ((payload ->> 'device'::text) = 'mobile'::text)
Heap Blocks: exact=1000
Buffers: shared hit=2 read=9
-> Bitmap Index Scan on idx_t7_events_device_btree (cost=0.00..94.67 rows=8317 width=0) (actual time=0.262..0.262 rows=8333 loops=1)
Index Cond: ((payload ->> 'device'::text) = 'mobile'::text)
Buffers: shared read=9
Planning:


In [33]:
%%sql

DROP TABLE IF EXISTS t7_hw_checkouts_norm;

CREATE TABLE t7_hw_checkouts_norm (
    event_id        INTEGER PRIMARY KEY,
    user_id         INTEGER NOT NULL,
    event_ts        TIMESTAMPTZ NOT NULL,
    order_id        TEXT NOT NULL,
    total           NUMERIC(10, 2) NOT NULL,
    items_count     INTEGER NOT NULL CHECK (items_count > 0),
    payment_method  TEXT NOT NULL CHECK (payment_method IN ('card', 'paypal', 'crypto'))
);

INSERT INTO t7_hw_checkouts_norm (
    event_id,
    user_id,
    event_ts,
    order_id,
    total,
    items_count,
    payment_method
)
SELECT
    event_id,
    user_id,
    event_ts,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total,
    (payload ->> 'items_count')::INTEGER AS items_count,
    payload ->> 'payment_method' AS payment_method
FROM t7_hw_events
WHERE event_type = 'checkout';

CREATE INDEX idx_t7_checkouts_norm_pm
    ON t7_hw_checkouts_norm (payment_method);

ANALYZE t7_hw_checkouts_norm;

SELECT COUNT(*) AS n_checkouts
FROM t7_hw_checkouts_norm;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

10000 rows affected.

1 rows affected.

n_checkouts
10000


In [34]:
%%sql

EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payload ->> 'payment_method' AS payment_method,
    AVG((payload ->> 'total')::NUMERIC) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

18 rows affected.

QUERY PLAN
Limit (cost=1954.62..1954.63 rows=3 width=72) (actual time=18.382..18.385 rows=3 loops=1)
Buffers: shared hit=1000
-> Sort (cost=1954.62..1968.47 rows=5542 width=72) (actual time=18.380..18.382 rows=3 loops=1)
Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
Sort Method: quicksort Memory: 25kB
Buffers: shared hit=1000
-> HashAggregate (cost=1799.86..1882.99 rows=5542 width=72) (actual time=18.345..18.367 rows=3 loops=1)
Group Key: (payload ->> 'payment_method'::text)
Batches: 1 Memory Usage: 217kB
Buffers: shared hit=1000


In [35]:
%%sql

EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payment_method,
    AVG(total) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

16 rows affected.

QUERY PLAN
Limit (cost=269.06..269.07 rows=3 width=46) (actual time=3.193..3.194 rows=3 loops=1)
Buffers: shared hit=94
-> Sort (cost=269.06..269.07 rows=3 width=46) (actual time=3.192..3.193 rows=3 loops=1)
Sort Key: (avg(total)) DESC
Sort Method: quicksort Memory: 25kB
Buffers: shared hit=94
-> HashAggregate (cost=269.00..269.04 rows=3 width=46) (actual time=3.181..3.183 rows=3 loops=1)
Group Key: payment_method
Batches: 1 Memory Usage: 24kB
Buffers: shared hit=94


## Reflection

**1. Плани до і після GIN.** Без індексу запит `payload @> {"payment_method":"card"}` виконувався через `Seq Scan`: cost 1756.61, 29.2 ms, фільтр відкинув 46 666 рядків заради 3 334 потрібних. Після `GIN (payload jsonb_path_ops)` розміром 1144 kB план став `Bitmap Index Scan` (0.46 ms, 4 буфери) + `Bitmap Heap Scan`: cost 1098.99, 11.2 ms. Виграш обмежений: `Heap Blocks: exact=1000` показує, що збіги розкидані по всіх 1000 сторінках, тож heap усе одно читається повністю.

**2. Чому `->>` не використовує GIN.** `jsonb_path_ops` підтримує лише оператори `@>`, `@?`, `@@`, а `payload ->> 'device' = 'mobile'` — звичайне порівняння тексту, яке планувальник не може зіставити з GIN. Тому навіть за наявності GIN був `Seq Scan` (8.2 ms) з поганою оцінкою: rows=250 замість 8 333. B-tree expression-індекс на `(payload ->> 'device')` дав `Bitmap Index Scan` за 1.3 ms і точну оцінку rows=8317, бо ANALYZE зібрав статистику по виразу. Альтернатива — переписати умову через `@>`.

**3. Read/write trade-off.** GIN пришвидшує containment- і key-existence-пошук по довільних ключах. Проте кожен INSERT/UPDATE розкладає payload на кілька записів індексу. Індекс займає місце (1144 kB на 50k подій) і додає роботи VACUUM. Для write-heavy event log це дорожчий ingest.

**4. JSONB чи нормалізація.** JSONB — для сирого event log зі змінною схемою: метадані експериментів, `model_version`, масиви `recommendations`. Нормалізована таблиця — для стабільних полів, з яких рахують фічі. Той самий GROUP BY по `payment_method` на `t7_hw_checkouts_norm` зайняв 3.2 ms і 94 буфери проти 18.5 ms і 1000 буферів на JSONB, а CHECK-обмеження гарантують якість даних.

**5. Новий event-type `refund`.** У JSONB достатньо INSERT з `event_type = 'refund'` і payload `{order_id, amount, reason}`. DDL не потрібен, GIN покриє нові ключі автоматично, але валідацію треба робити в застосунку або через CHECK. У normalized-підході потрібна міграція: `CREATE TABLE t7_hw_refunds_norm` з типізованими колонками, `CHECK (amount > 0)`, FK на `order_id` (спершу UNIQUE у checkouts), індекси, backfill і оновлення ETL.